# Multi-Probe Reward DPO — Qwen3.6-27B + FabricationGuard + ReasonGuard

**Notebook 35 · OpenInterp · 2026-04-29**

Proof-of-concept: fine-tune Qwen3.6-27B with **DPO using a combined multi-probe reward** built from two ProbeBench-registered probes:

- **FabricationGuard** at L31/end_question (factual hallucination)
- **ReasonGuard** at L55/mid_think (reasoning faithfulness)

## Why multi-probe matters

Goodfire RLFR (Apr 2026) proved single-probe RL works (-58% hallucination). Our extension: **two orthogonal probes simultaneously** — student must satisfy both, drastically harder to game than single-probe. Anti-Goodhart by orthogonal-objective construction, not just by training-time monitoring.

## Architecture (Goodfire RLFR pattern, multi-probe)

```
         Student (LoRA-trainable)         Frozen base (probe scorer)
prompt → Qwen3.6-27B + LoRA → gen   →    Qwen3.6-27B (no LoRA)
                                              │
                                              ▼
                                       L31 + L55 residuals
                                              │
                                              ▼
                                      FG probe + RG probe
                                              │
                                              ▼
                                  reward = -(0.5·P_FG + 0.5·P_RG)
                                              │
                          ◀── DPO update via preference pairs
```

**Key**: gradient never flows through probes. Student can only influence reward by generating different tokens — not by manipulating activations directly.

## Compute

- 1× RTX PRO 6000 96GB or H100 80GB
- ~2 hours wall-clock for the full POC
- ~$10-15 on Colab Pro+ credits


## 0. Drive mount + checkpoint dir (non-negotiable)


In [ ]:
# === DRIVE MOUNT — non-negotiable for any run >30min ===
from pathlib import Path
import os, sys

try:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
except Exception as e:
    print(f"Drive mount FAILED: {e}"); raise

DRIVE_ROOT = Path("/content/drive/MyDrive")
assert DRIVE_ROOT.exists(), "Drive mount silently failed"
NB_NAME = "35_multiprobe_dpo_poc"
OUT = DRIVE_ROOT / "openinterp_runs" / NB_NAME
OUT.mkdir(parents=True, exist_ok=True)
(OUT / "_dry_run.txt").write_text("drive mount OK")
print(f"✓ Drive checkpoint dir: {OUT}")
print(f"  Contents: {sorted(p.name for p in OUT.iterdir())}")


In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader || echo "no GPU"


## 1. Setup


In [ ]:
# IMPORTANT: torchao must be >=0.16.0 for peft compatibility on Qwen3.6-27B
%pip install -q -U torchao transformers accelerate peft trl datasets safetensors huggingface_hub
%pip install -q -U scikit-learn matplotlib joblib sentencepiece protobuf tqdm
print("installs done — RESTART RUNTIME if peft import fails, then run all cells again")


In [ ]:
import os, json, time, math, gc
from pathlib import Path
from typing import Optional, Tuple, List
from contextlib import contextmanager
import numpy as np, pandas as pd
import torch
import torch.nn.functional as F
from tqdm.auto import tqdm
import joblib
from huggingface_hub import login, hf_hub_download, HfApi
from datasets import load_dataset, Dataset
from sklearn.linear_model import LogisticRegressionCV
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
import matplotlib.pyplot as plt

CFG = {
    "model":            "Qwen/Qwen3.6-27B",
    "fg_repo":          "caiovicentino1/FabricationGuard-linearprobe-qwen36-27b",
    "rg_repo":          "caiovicentino1/ReasoningGuard-linearprobe-qwen36-27b",
    "fg_layer":         31,
    "rg_layer":         55,
    "probe_layers":     [31, 55],
    "corpus_n_simpleqa": 25,
    "corpus_n_gsm8k":   25,
    "cands_per_q":      4,
    "eval_n_simpleqa":  50,
    "eval_n_gsm8k":     50,
    "reward_alpha":     [0.5, 0.5],
    "lora_r":           16,
    "lora_alpha":       32,
    "dpo_lr":           5e-6,
    "dpo_beta":         0.1,
    "random_seed":      42,
    "output_repo":      "caiovicentino1/openinterp-multiprobe-dpo-poc",
}
THINK_OPEN_ID  = 248068
THINK_CLOSE_ID = 248069

torch.manual_seed(CFG["random_seed"]); np.random.seed(CFG["random_seed"])
import random; random.seed(CFG["random_seed"])

HF_TOKEN = os.environ.get("HF_TOKEN")
if HF_TOKEN is None:
    import getpass; HF_TOKEN = getpass.getpass("HF token (write scope): ")
login(HF_TOKEN, add_to_git_credential=False)

device = "cuda"; assert torch.cuda.is_available()
print(f"CUDA: {torch.cuda.get_device_name(0)}, {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")


## 2. Load Qwen3.6-27B + register hooks at L31, L55


In [ ]:
from transformers import AutoTokenizer, AutoModelForImageTextToText, AutoModelForCausalLM

print(f"Loading {CFG['model']} ...")
tok = AutoTokenizer.from_pretrained(CFG["model"], trust_remote_code=True)
try:
    model = AutoModelForImageTextToText.from_pretrained(
        CFG["model"], dtype=torch.bfloat16, attn_implementation="sdpa",
        device_map={"":device}, trust_remote_code=True)
except Exception:
    model = AutoModelForCausalLM.from_pretrained(
        CFG["model"], dtype=torch.bfloat16, attn_implementation="sdpa",
        device_map={"":device}, trust_remote_code=True)
model.eval()
for p in model.parameters(): p.requires_grad_(False)

def _block_list(m):
    candidates = [m, getattr(m,"model",None)]
    for s in candidates:
        if s is None: continue
        for path in [("model","language_model","layers"),("language_model","layers"),("model","layers"),("layers",)]:
            cur=s; ok=True
            for p in path:
                if hasattr(cur,p): cur=getattr(cur,p)
                else: ok=False; break
            if ok and hasattr(cur,"__getitem__"): return cur
    raise RuntimeError("layers not found")

blocks = _block_list(model)

class MultiLayerHook:
    def __init__(self, blocks, layers):
        self.bufs = {l:None for l in layers}; self.handles=[]
        for l in layers:
            self.handles.append(blocks[l].register_forward_hook(self._make(l)))
    def _make(self, l):
        def hook(_m,_i,out):
            h = out[0] if isinstance(out,tuple) else out
            self.bufs[l] = h.detach()
        return hook
    def pop(self, l):
        b = self.bufs[l]; self.bufs[l]=None; return b

ml_hook = MultiLayerHook(blocks, CFG["probe_layers"])
print(f"✓ Hooks registered at layers {CFG['probe_layers']}")


## 3. Load both probes


In [ ]:
def load_probe_bundle(repo, fname="probe.joblib"):
    p = hf_hub_download(repo, repo_type="dataset", filename=fname)
    obj = joblib.load(p)
    return obj["probe"], obj["scaler"]

fg_probe, fg_scaler = load_probe_bundle(CFG["fg_repo"])
rg_probe, rg_scaler = load_probe_bundle(CFG["rg_repo"])
print(f"✓ FabricationGuard L{CFG['fg_layer']}/end_question loaded")
print(f"✓ ReasonGuard       L{CFG['rg_layer']}/mid_think loaded")


## 4. Apply LoRA adapter


In [ ]:
from peft import LoraConfig, get_peft_model, TaskType

lora_cfg = LoraConfig(
    r=CFG["lora_r"], lora_alpha=CFG["lora_alpha"], lora_dropout=0.05,
    target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
    task_type=TaskType.CAUSAL_LM, bias="none",
)
model = get_peft_model(model, lora_cfg)
model.print_trainable_parameters()


## 5. Multi-probe scoring (frozen base via disable_adapter)


In [ ]:
@torch.no_grad()
def fg_score(question: str, answer: str) -> float:
    prompt = f"Q: {question}\nA: {answer}"
    enc = tok(prompt, return_tensors="pt", truncation=True, max_length=1024).to(device)
    n = int(enc["attention_mask"].sum().item())
    with model.disable_adapter():
        _ = model(**enc)
    h = ml_hook.pop(CFG["fg_layer"])[0, n-1].float().cpu().numpy()
    return float(fg_probe.predict_proba(fg_scaler.transform([h]))[0, list(fg_probe.classes_).index(1)])

@torch.no_grad()
def rg_score(question: str, full_answer: str) -> Optional[float]:
    chat = [{"role":"user","content":question}]
    prefix = tok.apply_chat_template(chat, tokenize=False,
                                     add_generation_prompt=True, enable_thinking=True)
    enc = tok(prefix + full_answer, return_tensors="pt", truncation=True, max_length=2048).to(device)
    ids = enc["input_ids"][0].tolist()
    op = next((i for i,t in enumerate(ids) if t == THINK_OPEN_ID), None)
    cl = next((i for i,t in enumerate(ids) if t == THINK_CLOSE_ID), None)
    if op is None or cl is None or cl <= op + 5: return None
    mid = (op + cl) // 2
    with model.disable_adapter():
        _ = model(**enc)
    h = ml_hook.pop(CFG["rg_layer"])[0, mid].float().cpu().numpy()
    return float(rg_probe.predict_proba(rg_scaler.transform([h]))[0, list(rg_probe.classes_).index(1)])

def combined_reward(q: str, a: str):
    fg = fg_score(q, a)
    rg = rg_score(q, a)
    if rg is not None:
        comb = CFG["reward_alpha"][0]*fg + CFG["reward_alpha"][1]*rg
    else:
        comb = fg
    return {"fg":fg,"rg":rg,"combined":comb,"reward":-comb,"has_think":rg is not None}


## 6. Mixed corpus + generation helper


In [ ]:
sqa = load_dataset("basicv8vc/SimpleQA", split="test").shuffle(seed=42).select(range(CFG["corpus_n_simpleqa"]))
gsm = load_dataset("openai/gsm8k", "main", split="test").shuffle(seed=42).select(range(CFG["corpus_n_gsm8k"]))
mixed = ([{"q":ex["problem"], "src":"simpleqa"} for ex in sqa] +
         [{"q":ex["question"],"src":"gsm8k"}    for ex in gsm])
np.random.default_rng(42).shuffle(mixed)
print(f"Mixed corpus: {len(mixed)} questions ({sum(1 for x in mixed if x['src']=='simpleqa')} factual, {sum(1 for x in mixed if x['src']=='gsm8k')} math)")

@torch.no_grad()
def gen_one(question: str, temp: float = 0.7, max_new: int = 256) -> str:
    msgs = [{"role":"user","content":question}]
    txt = tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True, enable_thinking=True)
    enc = tok(txt, return_tensors="pt").to(device)
    out = model.generate(**enc, max_new_tokens=max_new, do_sample=(temp>0),
                         temperature=temp if temp>0 else 1.0, top_p=0.9,
                         pad_token_id=tok.pad_token_id or tok.eos_token_id)
    return tok.decode(out[0, enc["input_ids"].shape[1]:], skip_special_tokens=True).strip()


## 7. Build DPO pairs (multi-probe reward picks chosen/rejected)


In [ ]:
pairs, telemetry = [], []
for ex in tqdm(mixed, desc="build DPO pairs"):
    q = ex["q"]
    cands = [gen_one(q, temp=0.7) for _ in range(CFG["cands_per_q"])]
    rs = [combined_reward(q, c) for c in cands]
    best_i  = int(np.argmin([r["combined"] for r in rs]))
    worst_i = int(np.argmax([r["combined"] for r in rs]))
    pairs.append({"prompt": q, "chosen": cands[best_i], "rejected": cands[worst_i]})
    telemetry.append({
        "q": q, "src": ex["src"],
        "fg_best": rs[best_i]["fg"], "fg_worst": rs[worst_i]["fg"],
        "rg_best": rs[best_i]["rg"], "rg_worst": rs[worst_i]["rg"],
        "combined_gap": rs[worst_i]["combined"] - rs[best_i]["combined"],
        "has_think_rate": sum(1 for r in rs if r["has_think"]) / len(rs),
    })
df_t = pd.DataFrame(telemetry)
df_t.to_csv(OUT / "pair_telemetry.csv", index=False)
print(f"\nMean combined_gap: {df_t['combined_gap'].mean():.3f}")
print(df_t.groupby("src")["combined_gap"].mean())

# Probe orthogonality check
df_both = df_t.dropna(subset=["rg_best"])
if len(df_both) > 5:
    rho_best  = np.corrcoef(df_both["fg_best"],  df_both["rg_best"])[0,1]
    rho_worst = np.corrcoef(df_both["fg_worst"], df_both["rg_worst"])[0,1]
    print(f"\nFG-RG Pearson: best={rho_best:+.3f}, worst={rho_worst:+.3f}")
    print(f"  ({'orthogonal ✅' if abs(rho_best)<0.4 else 'correlated ⚠️'})")


## 8. DPO training


In [ ]:
from trl import DPOTrainer, DPOConfig

ds = Dataset.from_list(pairs).train_test_split(test_size=0.2, seed=CFG["random_seed"])

dpo_cfg = DPOConfig(
    output_dir=str(OUT / "dpo_run"),
    num_train_epochs=1, per_device_train_batch_size=1,
    gradient_accumulation_steps=4, learning_rate=CFG["dpo_lr"], beta=CFG["dpo_beta"],
    save_steps=20, logging_steps=2, bf16=True, report_to="none",
    max_length=1024, max_prompt_length=512, save_strategy="steps", save_total_limit=2,
)
trainer = DPOTrainer(model=model, args=dpo_cfg,
                     train_dataset=ds["train"], eval_dataset=ds["test"],
                     processing_class=tok)
trainer.train()
trainer.save_model(str(OUT / "lora_final"))
print(f"✓ LoRA saved to {OUT / 'lora_final'}")


## 9. Evaluation — base vs student on held-out 100 queries


In [ ]:
sqa_e = load_dataset("basicv8vc/SimpleQA", split="test").shuffle(seed=99).select(range(CFG["eval_n_simpleqa"]))
gsm_e = load_dataset("openai/gsm8k", "main", split="test").shuffle(seed=99).select(range(CFG["eval_n_gsm8k"]))
eval_qs = ([(ex["problem"],"simpleqa")   for ex in sqa_e] +
           [(ex["question"],"gsm8k")     for ex in gsm_e])

results = []
for q, src in tqdm(eval_qs, desc="eval"):
    with model.disable_adapter():
        ans_b = gen_one(q, temp=0.0, max_new=512 if src=="gsm8k" else 256)
    ans_s = gen_one(q, temp=0.0, max_new=512 if src=="gsm8k" else 256)
    rb, rs_ = combined_reward(q, ans_b), combined_reward(q, ans_s)
    results.append({"q":q,"src":src,
                    "fg_base":rb["fg"],"fg_stud":rs_["fg"],
                    "rg_base":rb["rg"],"rg_stud":rs_["rg"],
                    "comb_base":rb["combined"],"comb_stud":rs_["combined"]})

df_e = pd.DataFrame(results); df_e.to_csv(OUT / "eval_results.csv", index=False)

print("\n" + "="*60)
print("  Multi-Probe DPO POC — Verdict")
print("="*60)
for src in ["simpleqa","gsm8k"]:
    sub = df_e[df_e.src==src]
    fg_red = (sub["fg_base"].mean()-sub["fg_stud"].mean())/sub["fg_base"].mean()*100
    rg_sub = sub.dropna(subset=["rg_base","rg_stud"])
    rg_red = (rg_sub["rg_base"].mean()-rg_sub["rg_stud"].mean())/rg_sub["rg_base"].mean()*100 if len(rg_sub)>0 else float("nan")
    cm_red = (sub["comb_base"].mean()-sub["comb_stud"].mean())/sub["comb_base"].mean()*100
    print(f"  {src:10s}: FG -{fg_red:.1f}% · RG -{rg_red:.1f}% · combined -{cm_red:.1f}%")
print("="*60)


## 10. HF push


In [ ]:
verdict = {
    "date": time.strftime("%Y-%m-%d"),
    "config": {k:v for k,v in CFG.items() if k!="output_repo"},
    "pairs_n": len(pairs),
    "eval_n": len(df_e),
    "mean_combined_gap_build": float(df_t["combined_gap"].mean()),
    "fg_rg_pearson_orthogonality": float(np.corrcoef(df_both["fg_best"], df_both["rg_best"])[0,1]) if len(df_both)>5 else None,
    "overall_combined_reduction": float((df_e["comb_base"].mean()-df_e["comb_stud"].mean())/df_e["comb_base"].mean()*100),
}
(OUT / "verdict.json").write_text(json.dumps(verdict, indent=2, default=str))

api = HfApi()
api.upload_folder(folder_path=str(OUT), repo_id=CFG["output_repo"],
                  repo_type="dataset", token=HF_TOKEN,
                  commit_message=f"Multi-Probe DPO POC results @ {time.strftime('%Y-%m-%d %H:%M')}")
print(f"\n✅ Pushed to https://huggingface.co/datasets/{CFG['output_repo']}")


## 11. Honest interpretation

Run **notebook 36 (Anti-Goodhart Fresh Probe Validation)** for the final verdict — fresh probes trained on student-generated samples will tell us whether the reduction is genuine or evasion.

If real: first OSS demonstration of multi-probe-reward DPO on 27B+ model. Direct extension of Goodfire RLFR (single-probe, -58% halu) to multi-probe orthogonal-objective design.
